In [ ]:
import cv2, numpy as np, yaml
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
from vista.qwen import get_model
from vista.utils_fun import IGNORE_CATEGORIES
from vista.pipeline.mypipeline import QwenCropCaptioner

cfg  = yaml.safe_load(open("./config/qwenyolo/mycfg.yaml"))
qcfg = cfg.get("qwen", {})

yolo= YOLO(cfg["yolo"]["model"])
qwen= get_model(cfg)
captioner = QwenCropCaptioner(qwen, system_prompt=qcfg.get("system_prompt"))

In [ ]:
def load_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, bgr = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"could not read frame {frame_idx}")
    return Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

video     = "./vista_videos/TrafficCollision_022 .mp4"   
frame_idx = 10
frame     = load_frame(video, frame_idx)
W, H      = frame.size
frame

In [ ]:
def detect_boxes(frame, conf=0.25):
    bgr = cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR)
    res = yolo.predict(bgr, conf=conf, verbose=False)[0]
    boxes = []
    for box, cls in zip(res.boxes.xyxy, res.boxes.cls):
        cat = res.names.get(int(cls.item()), "unknown")
        if cat in IGNORE_CATEGORIES:
            continue
        x1, y1, x2, y2 = box.cpu().numpy().tolist()
        boxes.append({"bbox": [x1, y1, x2, y2], "yolo_cat": cat})
    return boxes

boxes = detect_boxes(frame)
print(len(boxes), "boxes")

In [ ]:
def crop_with_margin(frame, bbox, frac, W, H, min_size=256):
    x1, y1, x2, y2 = bbox
    mx, my = (x2 - x1) * frac, (y2 - y1) * frac
    cx1, cy1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
    cx2, cy2 = min(W, int(x2 + mx)), min(H, int(y2 + my))
    if cx2 <= cx1 or cy2 <= cy1:
        return frame.crop((0, 0, 1, 1))
    crop = frame.crop((cx1, cy1, cx2, cy2))
    long_side = max(crop.size)
    if long_side < min_size:                       # solo upscaling, mai downscaling
        scale = min_size / long_side
        crop = crop.resize(
            (round(crop.width * scale), round(crop.height * scale)),
            Image.LANCZOS,
        )
    return crop

def caption_crops(crops, chunk=12):
    out = []
    for i in range(0, len(crops), chunk):
        out.extend(captioner(crops[i:i + chunk]))   # positional order preserved
    return out  

In [ ]:
def show_frame_with_boxes(frame, boxes):
    fig, ax = plt.subplots(figsize=(10, 10 * H / W))
    ax.imshow(frame)
    for i, b in enumerate(boxes):
        x1, y1, x2, y2 = b["bbox"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        fill=False, edgecolor="lime", lw=2))
        ax.text(x1, y1 - 4, str(i), color="lime", fontsize=12, weight="bold")
    ax.axis("off"); plt.show()

show_frame_with_boxes(frame, boxes)

In [ ]:
def compare_margins(frame, boxes, margins, W, H):
    crops_by_margin, labels_by_margin = {}, {}
    for m in margins:
        crops = [crop_with_margin(frame, b["bbox"], m, W, H) for b in boxes]
        crops_by_margin[m]  = crops
        labels_by_margin[m] = caption_crops(crops)

    n_obj, n_marg = len(boxes), len(margins)
    fig, axes = plt.subplots(n_obj, n_marg,
                             figsize=(3 * n_marg, 3 * n_obj), squeeze=False)
    for r in range(n_obj):
        for c, m in enumerate(margins):
            ax = axes[r][c]
            ax.imshow(crops_by_margin[m][r])
            lbl = labels_by_margin[m][r] or "(none)"
            ax.set_title(f"m={m} · {lbl}", fontsize=9)
            ax.axis("off")
    plt.tight_layout(); plt.show()

compare_margins(frame, boxes, margins=[0.0,0.5, 0.10, 0.25,0.27], W=W, H=H)